## Setup Environment

First, we need to install the required libraries for fine-tuning the TinyLlama model.

In [ ]:
import torch
import os

# Install libraries. suppress output for cleaner logs
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

# Check if CUDA is available for GPU training
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

# Mount Google Drive if you plan to store your dataset or model there
# from google.colab import drive
# drive.mount('/content/drive')

# If you uploaded your file directly to Colab's session storage, it will be in the /content/ directory.
# Example if your file is named 'my_training_data.jsonl':
# DATASET_PATH = "my_training_data.jsonl"
# You can verify its presence using:
# !ls -lh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.6 MB/s eta 0:00:00
CUDA available: True
CUDA device name: NVIDIA L4


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. Configuration
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_PATH = "fp_train.jsonl"        # <-- CHANGED: was all_combined.jsonl
EVAL_PATH    = "fp_val.jsonl"          # <-- NEW: held-out validation split
OUTPUT_DIR   = "./Qwen2.5-0.5b_fine_tuned"

COMPLETION_ONLY = True   # train loss on the assistant label only. Set False if your TRL errors.

# QLoRA configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training arguments
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 4
MAX_SEQ_LENGTH = 1024                  # <-- CHANGED: was 512 (avoid truncating labels)

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"bf16 supported: {USE_BF16} -> compute dtype: {COMPUTE_DTYPE}")

# 2. Load datasets
train_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
eval_dataset  = load_dataset("json", data_files=EVAL_PATH, split="train")
print(f"train={len(train_dataset)}  val={len(eval_dataset)}")

# 3. Model + tokenizer
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Custom ChatML template WITH {% generation %} tags so assistant_only_loss can
# build the label mask. The rendered text is identical to Qwen's stock ChatML,
# so it stays compatible with how Ollama serves the model at inference.
if COMPLETION_ONLY:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
        "{{ '<|im_start|>system\n' + message['content'] + '<|im_end|>\n' }}"
        "{% elif message['role'] == 'user' %}"
        "{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ '<|im_start|>assistant\n' }}{% generation %}{{ message['content'] + '<|im_end|>' }}{% endgeneration %}{{ '\n' }}"
        "{% endif %}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
)

# 4. Prepare for QLoRA
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",       # <-- CHANGED: was default (q_proj,v_proj only)
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 5. Training arguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim="paged_adamw_8bit",
    eval_strategy="epoch",             # <-- NEW
    save_strategy="epoch",
    load_best_model_at_end=True,       # <-- NEW
    metric_for_best_model="eval_loss", # <-- NEW
    greater_is_better=False,           # <-- NEW
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",        # <-- CHANGED: was constant
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    assistant_only_loss=COMPLETION_ONLY,   # <-- NEW
)

# 6. Train
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete!")

# 7. Save adapters
trainer.save_model(OUTPUT_DIR)
print(f"Fine-tuned LoRA adapters saved to {OUTPUT_DIR}")

bf16 supported: True -> compute dtype: torch.bfloat16


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

train=568  val=78


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Tokenizing eval dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Epoch,Training Loss,Validation Loss
1,0.151302,0.115905
2,0.168055,1.592240
3,0.146802,0.166113
4,0.052981,0.161189


Training complete!
Fine-tuned LoRA adapters saved to ./Qwen2.5-0.5b_fine_tuned


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL_ID     = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR  = "./Qwen2.5-0.5b_fine_tuned"        # adapters from the first run
OUTPUT_DIR   = "./Qwen2.5-0.5b_fine_tuned_v2"     # save the continued run here
DATASET_PATH = "all_combined.jsonl"
MAX_SEQ_LENGTH = 512

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map={"": 0},
)
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

# load the previously-trained adapters and make them trainable again
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=True)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,              # the 2 extra epochs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(model=model, train_dataset=dataset, args=training_args)
trainer.train()
trainer.save_model(OUTPUT_DIR)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
!pip uninstall -y torchao
!pip install -U peft transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "./tinyllama_fine_tuned"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

messages = [
    {
        "role": "system",
        "content": """ You are an assistant that extracts and normalizes a user's birth date from input text.

        STRICT_RULE:
          - Return only one value in format: MM-YYYY
          - Month must always be 2 digits (01–12)
          - If date is unclear return RETRY

        IMPORTANT:
          - Handle different date formats (DD/MM/YYYY, MM/YYYY, Month YYYY, etc.)
          - Handle common typos and spacing issues
          - Convert textual months to numeric format
          - Ignore extra words like "DOB", "born", etc.

        MAPPING_RULES:
          - Extract month and year from input → convert to MM-YYYY
          - If only year is present → RETRY
          - If multiple interpretations possible → RETRY

        Examples:
          - 12/05/1995 -> 05-1995
          - 05/1995 -> 05-1995
          - May 1995 -> 05-1995
          - may-1995 -> 05-1995
          - 1995 May -> 05-1995
          - DOB 5 1995 -> 05-1995
          - 5-95 -> 05-1995
          - septo 88 -> 07-1988
          - janoary eity two-> 01-82

          - 1995 -> RETRY
          - I am 30 years old -> RETRY"""
    },
    {
        "role": "user",
        "content": "septo 89"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


01-1989


In [ ]:
# code to download output folder in left panel
!zip -r Qwen2.5-0.5b_fine_tuned.zip Qwen2.5-0.5b_fine_tuned

  adding: Qwen2.5-0.5b_fine_tuned/ (stored 0%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/ (stored 0%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/adapter_config.json (deflated 59%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/adapter_model.safetensors (deflated 21%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/scheduler.pt (deflated 61%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/tokenizer.json (deflated 81%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/training_args.bin (deflated 53%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/chat_template.jinja (deflated 71%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/optimizer.pt (deflated 10%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/rng_state.pth (deflated 26%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/tokenizer_config.json (deflated 59%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/trainer_state.json (deflated 71%)
  adding: Qwen2.5-0.5b_fine_tuned/checkpoint-142/README.md (deflated 65

In [ ]:
import json, random
model.eval()
model.config.use_cache = True

def ask(system, user, max_new_tokens=16):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

rows = [json.loads(l) for l in open("fp_val.jsonl")]
random.seed(0); random.shuffle(rows)
correct = 0
N = 30
for r in rows[:N]:
    sys_m, usr_m, gold = (r["messages"][0]["content"],
                          r["messages"][1]["content"],
                          r["messages"][2]["content"])
    pred = ask(sys_m, usr_m)
    ok = (pred == gold)
    correct += ok
    print(f"{'OK ' if ok else 'XX '}{usr_m[:28]:30} gold={gold:18} pred={pred!r}")
print(f"\nexact-match on {N} val examples: {correct}/{N} = {correct/N:.0%}")

OK magi tier                      gold=magiTier           pred='magiTier'
OK agressive                      gold=8.5                pred='8.5'
OK not sure                       gold=RETRY              pred='RETRY'
OK yeah i bought it               gold=true               pred='true'
OK 1                              gold=1                  pred='1'
OK go ahead                       gold=yes                pred='yes'
XX illustrason for ltc            gold=ltc                pred='viewLtcIllustrations'
OK safe                           gold=4.5                pred='4.5'
OK change                         gold=yes                pred='yes'
XX blah blah                      gold=NONE               pred='{"cStage": "user", "cState": null}'
OK change my birthdate            gold={"cStage": "user", "cState": "birthdate"} pred='{"cStage": "user", "cState": "birthdate"}'
OK cancel                         gold=no                 pred='no'
OK healthcare exchange            gold=preMedicarePlan    